# Task 3: Retrieval (RQ3) + Stance (RQ4), Kaggle atau Colab
Notebook ini jalan di **Kaggle** maupun **Google Colab** (otomatis mendeteksi platform di sel kedua).
Kode ditarik dari **GitHub**, checkpoint NER dan data tambahan dari **Google Drive** (`gdown`), Komdigi dari Kaggle Dataset (Kaggle) atau Drive tim (Colab).
Folder Drive harus dibagikan "Anyone with the link" (Viewer).

Sebelum mulai:
1. **Kaggle**: Settings → Accelerator GPU T4 x2 / P100, Internet On. **Colab**: Runtime → Change runtime type → T4 GPU.
2. (Kaggle, opsional) Add Input → dataset `ireddragonicy/indonesian-hoax-news-dataset` (v396). Tanpa ini, Komdigi diunduh otomatis dari Drive tim.
3. (Repo privat saja) Secret `GITHUB_TOKEN`: Kaggle → Add-ons → Secrets; Colab → ikon kunci di sidebar kiri.
4. (Opsional, BRIN) Secret `HF_TOKEN` dengan cara yang sama.
5. Kode task 3 harus sudah ada di `GITHUB_BRANCH` (default `main`).
6. Hasil: Kaggle → tab Output; Colab → terunduh otomatis di sel terakhir (cadangan: disalin ke Drive Anda).

In [1]:
# ===== KONFIGURASI: ubah di sini =====
GITHUB_REPO   = "danenftyessir/indonesian-hoax-checker"
GITHUB_BRANCH = "main"
# Folder Drive tim (dokumen). Hanya subfolder `data/` dan `artifacts/` di dalamnya yang disalin ke repo.
GDRIVE_DATA_FOLDER_ID = "1rregnX8Qq9pU7rlTB7QUiXubWVJ5-zS0"
# Folder hasil_training: berisi ner_<waktu>/ner_results.zip (checkpoint NER dari Jonathan)
GDRIVE_NER_FOLDER_ID  = "1Eq9C3SIW42xYpRn5UVAwoiLrRkOY52Pn"
NER_CHECKPOINT = "artifacts/checkpoints/ner/ner_indobert_seed42"
RUN_STAGE1_TRAINING = True   # fine-tune IndoBERT di IndoNLI (~20-40 menit di T4)

In [2]:
import os, subprocess, glob, shutil, pathlib, sys

ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else ("/content" if ON_COLAB else os.path.abspath("work"))
os.makedirs(WORK, exist_ok=True)
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "lokal", "| WORK =", WORK)

def get_secret(name):
    """Ambil secret dari Kaggle Secrets atau Colab Secrets; None bila tidak ada."""
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

token = get_secret("GITHUB_TOKEN")
url = f"https://{token+'@' if token else ''}github.com/{GITHUB_REPO}.git"
REPO = f"{WORK}/indonesian-hoax-checker"
os.chdir(WORK)   # keluar dari repo lama dulu; cwd yang terhapus membuat git clone gagal (exit 128)
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","-b",GITHUB_BRANCH,url,REPO],check=True)
os.chdir(REPO)
!git log --oneline -3

platform: Colab | WORK = /content
de27e77 (grafted, HEAD -> main, origin/main, origin/HEAD) fix(notebook): make Task 3 notebook run on both Kaggle and Colab


In [3]:
# extras [training] = datasets, accelerate, seqeval, pytorch-crf (dibutuhkan ner.predict untuk entitas)
!pip -q install -e ".[training]" gdown 2>&1 | tail -3
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "GPU belum aktif: Kaggle Settings -> Accelerator, atau Colab Runtime -> Change runtime type -> T4 GPU"

cuda: True Tesla T4


## Data
Komdigi dari Kaggle Input (bila ada) atau Drive tim, IndoNLI dari GitHub raw, tambahan (BRIN, checkpoint, dll.) dari Drive.

In [4]:
import yaml, gdown
os.makedirs("data/raw/refuted_claims", exist_ok=True)
DEST = "data/raw/refuted_claims/komdigi_hoaks.csv"
# File Komdigi di Drive tim: <folder>/data/archive/komdigi_hoaks.csv (Anyone with the link, Viewer)
KOMDIGI_DRIVE_FILE_ID = "1nsRT_9zvGbizx8H-sBSbDG1-vtyfPsp_"

# 1) Kaggle Input (rekursif: dataset bisa berada di sub-folder, mis. /kaggle/input/datasets/<owner>/<nama>/); di Colab kosong
hits = glob.glob("/kaggle/input/**/komdigi_hoaks.csv", recursive=True) if ON_KAGGLE else []
if hits:
    shutil.copy(hits[0], DEST); print("Komdigi dari Kaggle Input:", hits[0])
else:
    # 2) Cadangan: unduh langsung dari Drive tim (tanpa Add Input)
    print("komdigi_hoaks.csv tidak ada di Kaggle Input (atau bukan Kaggle), unduh dari Drive...")
    gdown.download(id=KOMDIGI_DRIVE_FILE_ID, output=DEST, quiet=False)
    assert os.path.exists(DEST) and os.path.getsize(DEST) > 1_000_000, (
        "Unduhan Komdigi dari Drive gagal/terlalu kecil. Pastikan file Drive 'Anyone with the link' "
        "(Kaggle: boleh juga Add Input -> ireddragonicy/indonesian-hoax-news-dataset v396).")
n_rows = sum(1 for _ in open(DEST, encoding="utf-8-sig")) - 1
print("baris data (kasar, termasuk baris lanjutan):", n_rows)

src = yaml.safe_load(open("configs/data_sources.yaml"))["stance"]["indonli"]
os.makedirs(src["local_raw"], exist_ok=True)
for f in src["files"]:
    !wget -q -nc -P {src["local_raw"]} {src["base_url"]}{f}
!ls -la data/raw/refuted_claims data/raw/stance/indonli

komdigi_hoaks.csv tidak ada di Kaggle Input (atau bukan Kaggle), unduh dari Drive...


Downloading...
From: https://drive.google.com/uc?id=1nsRT_9zvGbizx8H-sBSbDG1-vtyfPsp_
To: /content/indonesian-hoax-checker/data/raw/refuted_claims/komdigi_hoaks.csv
100%|██████████| 10.1M/10.1M [00:00<00:00, 82.1MB/s]


baris data (kasar, termasuk baris lanjutan): 3048
data/raw/refuted_claims:
total 9848
drwxr-xr-x 2 root root     4096 Nov 13 10:47 .
drwxr-xr-x 6 root root     4096 Nov 13 10:47 ..
-rw-r--r-- 1 root root        0 Nov 13 10:47 .gitkeep
-rw-r--r-- 1 root root 10073039 Nov 11 13:55 komdigi_hoaks.csv

data/raw/stance/indonli:
total 6832
drwxr-xr-x 2 root root    4096 Nov 13 10:47 .
drwxr-xr-x 3 root root    4096 Nov 13 10:47 ..
-rw-r--r-- 1 root root       0 Nov 13 10:47 .gitkeep
-rw-r--r-- 1 root root 1608272 Nov 13 10:47 test_expert.jsonl
-rw-r--r-- 1 root root  796010 Nov 13 10:47 test_lay.jsonl
-rw-r--r-- 1 root root 3788819 Nov 13 10:47 train.jsonl
-rw-r--r-- 1 root root  784776 Nov 13 10:47 val.jsonl


In [5]:
import gdown, zipfile

if GDRIVE_DATA_FOLDER_ID:
    try:
        gdown.download_folder(id=GDRIVE_DATA_FOLDER_ID, output=f"{WORK}/gdrive_in", quiet=True)
        for rel in ["data", "artifacts"]:
            p = pathlib.Path(f"{WORK}/gdrive_in")/rel
            if p.exists(): shutil.copytree(p, rel, dirs_exist_ok=True); print("disalin:", rel)
    except Exception as e:
        print("Folder data Drive tidak bisa diunduh (bukan 'Anyone with the link' / file dibatasi pemilik); dilewati. "
              "Detail:", str(e).strip().splitlines()[0][:120])

# Checkpoint NER: pilih ner_results.zip dari subfolder ner_* terbaru, unduh satu file saja
if GDRIVE_NER_FOLDER_ID and not os.path.exists(NER_CHECKPOINT):
    try:
        files = gdown.download_folder(id=GDRIVE_NER_FOLDER_ID, skip_download=True, quiet=True, remaining_ok=True)
        zips = sorted([f for f in files if f.path.replace(chr(92), "/").endswith("/ner_results.zip")
                       and "/rq1_" not in f.path.replace(chr(92), "/")], key=lambda f: f.path)
        assert zips, "ner_results.zip tidak ditemukan di folder hasil_training"
        z = zips[-1]; print("memakai", z.path)
        gdown.download(id=z.id, output=f"{WORK}/ner_results.zip", quiet=False)
        zipfile.ZipFile(f"{WORK}/ner_results.zip").extractall(".")
    except Exception as e:
        print("Checkpoint NER gagal diunduh; ablasi filter entitas dilewati:", e)
print("checkpoint NER ada:", os.path.exists(NER_CHECKPOINT))

Folder data Drive tidak bisa diunduh (bukan 'Anyone with the link' / file dibatasi pemilik); dilewati. Detail: Failed to retrieve file url:
memakai ner_20261112-1801/ner_results.zip


Downloading...
From: https://drive.google.com/uc?id=1mhTumyRqf0PfxW86Voadx06Im4YnH1dl
To: /content/ner_results.zip
100%|██████████| 260k/260k [00:00<00:00, 114MB/s]

checkpoint NER ada: False


## Fase 1: klaim, split, konversi stance

In [6]:
!python -m indonesian_hoax_checker.retrieval.corpus data/raw/refuted_claims/komdigi_hoaks.csv data/interim/refuted_claims/claims_raw.jsonl
!python -m indonesian_hoax_checker.evaluation.splits data/interim/refuted_claims/claims_raw.jsonl --kaggle-version 396
for s in ["train","val","test_lay","test_expert"]:
    !python -m indonesian_hoax_checker.stance.dataset indonli data/raw/stance/indonli/{s}.jsonl data/processed/stance/indonli_{s}.jsonl --split-name {s}

# Anotasi tim (kueri retrieval, pasangan stance) mengacu ke claim_id Komdigi v396. Bila versi Kaggle Input
# berbeda, gold id bisa hilang dan metrik menyesatkan: hentikan di sini.
import json
ids = {json.loads(l)["claim_id"] for l in open("data/processed/refuted_claims/claims.jsonl", encoding="utf-8")}
gold = {g for l in open("data/annotations/claim_verification/retrieval_queries.jsonl", encoding="utf-8") if l.strip()
        for g in json.loads(l)["gold_claim_ids"]}
missing = gold - ids
assert not missing, f"{len(missing)} gold_claim_id tidak ada di basis klaim: versi dataset Kaggle Input berbeda dari v396"
print("gold id cocok:", len(gold))

{
  "input_rows": 3048,
  "written": 3048,
  "n_claims": 3048,
  "with_claim_sentence": 3048,
  "with_fact_sentence": 2948,
  "with_tags": 3048,
  "empty_clarification": 0,
  "date_min": "2024-12-27T23:00:00+00:00",
  "date_max": "2026-11-09T23:06:00+00:00",
  "title_tags": {
    "HOAKS": 3048
  }
}
{
  "n_claims": 3048,
  "threshold": 0.8,
  "n_pairs_above_threshold": 63,
  "n_clusters": 2989,
  "n_clusters_multi": 48,
  "n_claims_in_multi_clusters": 107,
  "largest_cluster_sizes": [
    4,
    4,
    4,
    3,
    3
  ],
  "split_counts": {
    "train": 2398,
    "dev": 299,
    "test": 351
  },
  "test_start": "2026-09-01",
  "dev_start": "2026-07-01",
  "kaggle_version": "396",
  "source_claims": "data/interim/refuted_claims/claims_raw.jsonl"
}
{
  "written": 10330,
  "labels": {
    "supports": 3476,
    "refutes": 3439,
    "unrelated": 3415
  },
  "negation_rate": {
    "supports": 0.058,
    "refutes": 0.545,
    "unrelated": 0.088
  }
}
{
  "written": 2197,
  "labels": {
    "

## RQ3: retrieval (BM25 vs E5, tanpa/dengan filter entitas)

In [7]:
import json
Q = "data/annotations/claim_verification/retrieval_queries.jsonl"
_rows = [json.loads(l) for l in open(Q, encoding="utf-8") if l.strip()]
_empty = [r for r in _rows if not (r["query"] or "").strip()]
print(f"kueri: {len(_rows)}, kosong: {len(_empty)} ({sorted({r['type'] for r in _empty})})")
if any(r["type"] == "paraphrase" for r in _empty):
    print("PERINGATAN: parafrasa belum ditulis. Kueri kosong dilewati evaluasi; yang tersisa adalah judul asli "
          "(otomatis ketemu 100% karena judulnya ada di basis) dan no_match. JANGAN dipakai di laporan.")
have_ner = os.path.exists(NER_CHECKPOINT)
if have_ner:
    # entitas PREDIKSI NER (keputusan 0005) untuk basis klaim dan kueri
    !python -m indonesian_hoax_checker.retrieval.annotate_entities --checkpoint {NER_CHECKPOINT} --queries {Q}
runs = [("rq3_bm25_nofilter","configs/models/retrieval_bm25.yaml",""),
        ("rq3_e5_nofilter","configs/models/retrieval_embeddings.yaml","")]
if have_ner:
    runs += [("rq3_bm25_entfilter","configs/models/retrieval_bm25.yaml","--entity-filter"),
             ("rq3_e5_entfilter","configs/models/retrieval_embeddings.yaml","--entity-filter")]
for name,cfg,flag in runs:
    print("=== ",name)
    r = subprocess.run(f"python -m indonesian_hoax_checker.retrieval.evaluate {cfg} --queries {Q} --run-name {name} {flag}",shell=True)
    if r.returncode: print("GAGAL:", name)

kueri: 300, kosong: 100 (['paraphrase'])
PERINGATAN: parafrasa belum ditulis. Kueri kosong dilewati evaluasi; yang tersisa adalah judul asli (otomatis ketemu 100% karena judulnya ada di basis) dan no_match. JANGAN dipakai di laporan.
===  rq3_bm25_nofilter
===  rq3_e5_nofilter


## RQ4: stance

In [8]:
if RUN_STAGE1_TRAINING:
    !python -m indonesian_hoax_checker.stance.train configs/models/stance_nli.yaml --train data/processed/stance/indonli_train.jsonl --dev data/processed/stance/indonli_val.jsonl --output artifacts/checkpoints/stance_indonli --seed 42

config.json: 100% 1.53k/1.53k [00:00<00:00, 5.66MB/s]
tokenizer_config.json: 100% 2.00/2.00 [00:00<00:00, 10.5kB/s]
vocab.txt: 100% 229k/229k [00:00<00:00, 73.0MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 527kB/s]
Map: 100% 10330/10330 [00:01<00:00, 7060.10 examples/s]
Map: 100% 2197/2197 [00:00<00:00, 6851.99 examples/s]
[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.

pytorch_model.bin: downloading bytes:  91% 455M/498M [00:03<00:00, 169MB/s, 38.1MB/s  ]
pytorch_model.bin: reconstructing file:  54% 268M/498M [00:03<00:02, 77.6MB/s]
pytorch_model.bin: reconstructing file:  81% 402M/498M [00:03<00:00, 118MB/s, 24.6MB/s  ]
pytorch_model.bin: downloading bytes: 100% 472M/472M [00:04<00:00, 110MB/s, 39.7MB/s  ]
pytorch_model.bin: reconstructing file: 100% 498M/498M [00:04<00:00, 116MB/s, 43.3MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 20429.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobe

## RQ4 tahap 2: BRIN + pasangan tim (otomatis dilewati bila datanya belum ada)
BRIN hanya berlabel `refutes`; dilatih sendirian model akan runtuh ke satu label. Jadi tahap 2 melatih
**IndoNLI train + BRIN** bersama, mulai dari checkpoint tahap 1. Pasangan tim **tidak dipakai untuk melatih**:
dipakai khusus sebagai set uji (RQ4), dan hanya baris `status=verified` dengan dua annotator.

In [9]:
import json, os
STAGE2_OK = False
BRIN_CSV = "data/raw/stance/brin_fakenews_mafindo.csv"

# BRIN: dari folder Drive (data/raw/stance/...) atau HuggingFace (dataset gated; butuh secret HF_TOKEN)
if not os.path.exists(BRIN_CSV):
    try:
        hf = get_secret("HF_TOKEN")
        assert hf, "secret HF_TOKEN tidak ada"
        from datasets import load_dataset
        ds = load_dataset("nlp-brin-id/fakenews-mafindo", token=hf)
        os.makedirs(os.path.dirname(BRIN_CSV), exist_ok=True)
        ds[list(ds.keys())[0]].to_csv(BRIN_CSV, index=False)
    except Exception as e:
        print("BRIN tidak tersedia (minta akses HF + secret HF_TOKEN, atau taruh CSV di Drive):", repr(e)[:200])

if os.path.exists(BRIN_CSV) and os.path.exists("artifacts/checkpoints/stance_indonli"):
    !python -m indonesian_hoax_checker.stance.dataset brin {BRIN_CSV} data/processed/stance/brin_refutes.jsonl
    !python -m indonesian_hoax_checker.stance.train configs/models/stance_nli.yaml --train data/processed/stance/indonli_train.jsonl data/processed/stance/brin_refutes.jsonl --dev data/processed/stance/indonli_val.jsonl --init artifacts/checkpoints/stance_indonli --output artifacts/checkpoints/stance_final --seed 42 --epochs 1
    STAGE2_OK = os.path.exists("artifacts/checkpoints/stance_final")
print("tahap 2 dijalankan:", STAGE2_OK)

BRIN tidak tersedia (minta akses HF + secret HF_TOKEN, atau taruh CSV di Drive): AssertionError('secret HF_TOKEN tidak ada')
tahap 2 dijalankan: False


In [10]:
# Set uji pasangan tim: hanya yang sudah diverifikasi dua orang (stance_pairs.jsonl lengkap berstatus draft sampai diisi)
src_pairs = "data/annotations/claim_verification/stance_pairs.jsonl"
verified = [json.loads(l) for l in open(src_pairs, encoding="utf-8") if l.strip()]
verified = [r for r in verified if r.get("status") == "verified" and r.get("hypothesis")]
TEAM_TEST = None
if len(verified) >= 30:
    with open("data/processed/stance/team_verified.jsonl", "w", encoding="utf-8") as f:
        for r in verified: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    !python -m indonesian_hoax_checker.stance.dataset team data/processed/stance/team_verified.jsonl data/processed/stance/team_test.jsonl
    TEAM_TEST = "team_test"
print(f"pasangan tim terverifikasi: {len(verified)} ->", TEAM_TEST or "dilewati (butuh >= 30; target 300)")

pasangan tim terverifikasi: 0 -> dilewati (butuh >= 30; target 300)


In [11]:
models = [("rq4_similarity", "similarity"),
          ("rq4_mdeberta_zeroshot", "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")]
if os.path.exists("artifacts/checkpoints/stance_indonli"): models.append(("rq4_indobert_indonli", "artifacts/checkpoints/stance_indonli"))
if os.path.exists("artifacts/checkpoints/stance_final"):   models.append(("rq4_indobert_final", "artifacts/checkpoints/stance_final"))
splits = ["indonli_test_expert", "indonli_val"] + ([TEAM_TEST] if TEAM_TEST else [])
for split in splits:
    for name, m in models:
        if m == "similarity":
            !python -m indonesian_hoax_checker.stance.predict --model similarity --fit-threshold-on data/processed/stance/indonli_val.jsonl --pairs data/processed/stance/{split}.jsonl --run-name {name}_{split}
        else:
            !python -m indonesian_hoax_checker.stance.predict --model {m} --pairs data/processed/stance/{split}.jsonl --run-name {name}_{split}

{
  "n": 2984,
  "macro_f1": 0.3156,
  "per_class": {
    "supports": {
      "precision": 0.4091,
      "recall": 0.415,
      "f1": 0.412,
      "support": 1041
    },
    "refutes": {
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0,
      "support": 999
    },
    "unrelated": {
      "precision": 0.3983,
      "recall": 0.8136,
      "f1": 0.5348,
      "support": 944
    }
  },
  "confusion": {
    "labels": [
      "supports",
      "refutes",
      "unrelated"
    ],
    "matrix": [
      [
        432,
        0,
        609
      ],
      [
        448,
        0,
        551
      ],
      [
        176,
        0,
        768
      ]
    ],
    "supports_as_refutes": 0,
    "refutes_as_supports": 448,
    "direction_swap_rate": 0.2196
  },
  "with_negation": {
    "n": 392,
    "macro_f1": 0.2175,
    "per_class": {
      "supports": {
        "precision": 0.262,
        "recall": 0.4537,
        "f1": 0.3322,
        "support": 108
      },
      "refutes": {
 

In [12]:
import json
for f in sorted(glob.glob("outputs/metrics/*.json")):
    print(f); print(json.dumps(json.load(open(f)),indent=1,ensure_ascii=False)[:800])

outputs/metrics/ner_bilstm_crf_claim_time_seed13.json
{
 "name": "ner_bilstm_crf_claim_time",
 "model": "bilstm_crf",
 "seed": 13,
 "device": "cuda",
 "git_commit": "0008258",
 "best_epoch": 14,
 "data": {
  "train": "data/processed/ner/seed_train.conll",
  "dev": "data/processed/ner/seed_dev.conll",
  "test": "data/processed/ner/seed_test.conll"
 },
 "dev": {
  "precision": 0.734375,
  "recall": 0.7580645161290323,
  "f1": 0.7460317460317459,
  "per_type": {
   "CLAIM": {
    "precision": 0.7755102040816326,
    "recall": 0.7755102040816326,
    "f1": 0.7755102040816326,
    "support": 49
   },
   "LOC": {
    "precision": 0.5384615384615384,
    "recall": 0.6363636363636364,
    "f1": 0.5833333333333334,
    "support": 11
   },
   "ORG": {
    "precision": 0.6666666666666666,
    "recall": 0.5714285714285714,
    "f1": 0.6153846153846153,
 
outputs/metrics/ner_bilstm_crf_claim_time_seed2026.json
{
 "name": "ner_bilstm_crf_claim_time",
 "model": "bilstm_crf",
 "seed": 2026,
 "device":

## Simpan hasil
Notebook men-zip hasil ke `task3_results.zip`.
- **Colab**: file diunduh otomatis lewat browser; bila gagal, disalin ke Google Drive Anda (`/MyDrive/task3_results/`).
- **Kaggle**: tab *Output* → Download (bila unduhan Kaggle bermasalah, jalankan di Colab).

`claims_entities.jsonl` dan `retrieval_queries.jsonl` (entitas terisi) ada di dalam zip: salin kembali ke repo lokal agar ablasi bisa direproduksi.

In [13]:
ZIP = f"{WORK}/task3_results.zip"
if os.path.exists(ZIP): os.remove(ZIP)
!cd {WORK} && zip -rq task3_results.zip indonesian-hoax-checker/outputs indonesian-hoax-checker/artifacts/checkpoints/stance_indonli indonesian-hoax-checker/artifacts/checkpoints/stance_final indonesian-hoax-checker/data/splits indonesian-hoax-checker/data/processed/refuted_claims/claims_entities.jsonl indonesian-hoax-checker/data/annotations/claim_verification/retrieval_queries.jsonl || true
!ls -lh {ZIP}

if ON_COLAB:
    # 1) unduhan langsung lewat browser; 2) cadangan: salin ke Drive pribadi (butuh izin popup)
    try:
        from google.colab import files
        files.download(ZIP)
    except Exception as e:
        print("Unduhan browser gagal:", repr(e)[:150])
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs("/content/drive/MyDrive/task3_results", exist_ok=True)
        shutil.copy(ZIP, "/content/drive/MyDrive/task3_results/")
        print("Disalin ke Drive: MyDrive/task3_results/task3_results.zip")
    except Exception as e:
        print("Salin ke Drive dilewati:", repr(e)[:150])
elif ON_KAGGLE:
    print("Kaggle: ambil dari tab Output (atau Save Version lalu Download).")

-rw-r--r-- 1 root root 1.6G Nov 13 10:57 /content/task3_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mounted at /content/drive
Disalin ke Drive: MyDrive/task3_results/task3_results.zip
